In [1]:
import shapely.geometry as geometry
from shapely.ops import linemerge, unary_union, polygonize
from shapely.geometry import mapping

import networkx as nx
import pandas as pd
import numpy as np
import pickle

import matplotlib.pyplot as plt

from haversine import haversine

from convenient_pickle import *

import os
import time
import gc

safe_cwd = os.getcwd()

%config InteractiveShell.cache_size = 0

# Load in relevant data

In [2]:
def load_pickles(prefix):
    G = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_graph.pkl')
    total_result_nodes = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_total_result_nodes.pkl')
    total_result_ways = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_total_result_ways.pkl')
    used_bboxes = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_used_bboxes.pkl')
    return G, total_result_nodes, total_result_ways, used_bboxes

def lite_load_pickles(prefix):
    G = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_graph.pkl')
    total_result_ways = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_total_result_ways.pkl')
    return G, total_result_ways

def make_ways_lighter(total_result_ways) :
    ways_dict = dict()
    ways_dict['ways'] = dict()
    ways_dict['nodes'] = dict()
    for way in total_result_ways: 
        way_id = way.id
        node_ids = []
        for node in way.nodes: 
            node_ids.append(node.id)
            ways_dict[node.id] = {'lat': float(node.lat),'lon': float(node.lon)}
        ways_dict['ways'][way_id] = node_ids
    return ways_dict
            

In [3]:
def get_edge_distance(node1, node2): 
    lat1 = float(node1['lat'])
    lon1 = float(node1['lon'])
    lat2 = float(node2['lat'])
    lon2 = float(node2['lon'])

    return haversine((lat1,lon1), (lat2, lon2))


In [4]:
def convert_to_borders(ways):
    lss = [] 
    
    for ii_w,way in enumerate(ways):
        ls_coords = []
    
        for node in way.nodes:
            ls_coords.append((node.lon,node.lat)) 
    
        lss.append(geometry.LineString(ls_coords))
    
    
    merged = linemerge([*lss]) 
    borders = unary_union(merged) # linestrings to a MultiLineString
    polygons = list(polygonize(borders))
    return merged, borders, polygons

def convert_to_dispersed_borders(ways):
    lss = [] 
    
    for ii_w,way in enumerate(ways):
        ls_coords = []
    
        for node in way.nodes:
            ls_coords.append((node.lon,node.lat)) 
    
        lss.append(geometry.LineString(ls_coords))
    
    
    merged = linemerge([*lss]) 
    return merged
    
#See how many ways each node belongs to
def collect_overlap(ways):
    outdict = dict()
    for way in ways: 
        way_nodes = way.nodes
        for node in way_nodes: 
            if node.id not in outdict.keys(): 
                outdict[node.id] = 1
            else: 
                outdict[node.id] += 1
    outdict = [(i, outdict[i]) for i in outdict.keys()]
    outdict = sorted(outdict, key = lambda x: -x[1])
    return outdict

def make_graph(ways):
    G = nx.Graph()
    
    for way in ways: 
        for node_dex in range(len(way.nodes)): 
            node = way.nodes[node_dex]
            node_id = node.id
            if G.has_node(node_id) == False:
                G.add_node(node_id)
            if node_dex > 0: 
                source = {'lat':node.lat, 'lon':node.lon}
                target = {'lat':way.nodes[node_dex-1].lat, 'lon':way.nodes[node_dex-1].lon}
                edge_distance = get_edge_distance(source, target)
                G.add_edge(node_id,way.nodes[node_dex-1].id,weight=edge_distance)
                
    return G

def make_new_lcc(G, ways):
    cclist = sorted([i for i in nx.connected_components(G)], key = lambda x: -len(x))
    lcc = cclist[0]
    lcc_ways = []
    for way in ways: 
        for node in way.nodes: 
            if node.id in lcc: 
                lcc_ways.append(way)
                break
    lcc_node_set = set()
    for way in lcc_ways:
        for node in way.nodes:
            lcc_node_set.add(node.id)
    return pd.Series(lcc_ways)

# Serialize your merged network to GeoJSON
def quick_map(lcc_merged):
    bike_geojson = mapping(lcc_merged)
    
    # Illinois center
    m = folium.Map(location=[40.0, -89.2], zoom_start=6, tiles='CartoDB positron')
    
    # Illinois outline
    folium.GeoJson(
        "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json",
        name="Illinois",
        style_function=lambda f: {
            'fillColor': '#E6F1FB',
            'color': '#185FA5',
            'weight': 1.5,
            'fillOpacity': 0.25
        } if f['properties']['name'] == 'Illinois' else {
            'fillOpacity': 0,
            'color': 'none',
            'weight': 0
        }
    ).add_to(m)
    
    # Bike/pedestrian network
    folium.GeoJson(
        bike_geojson,
        name="Bike network",
        style_function=lambda f: {
            'color': '#1D9E75',
            'weight': 2,
            'opacity': 0.85
        }
    ).add_to(m)
    
    folium.Rectangle(
        bounds=[[bbox[0], bbox[1]], [bbox[2], bbox[3]]],
        color='#E24B4A',
        weight=2,
        fill=False
    ).add_to(m)
    
    # new_bbox = get_new_bbox(bbox, interval, 'northeast')
    
    # folium.Rectangle(
    #     bounds=[[new_bbox[0], new_bbox[1]], [new_bbox[2], new_bbox[3]]],
    #     color='#E24B4A',
    #     weight=2,
    #     fill=False
    # ).add_to(m)
    
    
    # Zoom to the network
    bounds = lcc_merged.bounds  # (minx, miny, maxx, maxy)
    m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
    
    folium.LayerControl().add_to(m)
    return m

def break_down_into_nodes(ways): 
    node_list = []
    for way in ways: 
        node_list += way.nodes
    return node_list

def get_node_ids(nodelist):
    id_list = []
    for node in nodelist: 
        id_list.append(node.id)
    return id_list

def get_way_node_ids(ways): 
    node_list = break_down_into_nodes(ways)
    id_list = get_node_ids(node_list)
    return id_list

def average_degree(subgraph): 
    average = 0
    for node in subgraph.nodes: 
        average += subgraph.degree(node)
    return average/len(subgraph.nodes)

def get_node_latlon_dict(ways): 
    outdict = dict()
    nodelist = break_down_into_nodes(ways)
    for node in nodelist: 
        outdict[node.id] = {'lat': node.lat, 'lon': node.lon}
    return outdict

def get_edge_distance(node1, node2): 
    lat1 = float(node1['lat'])
    lon1 = float(node1['lon'])
    lat2 = float(node2['lat'])
    lon2 = float(node2['lon'])

    return haversine((lat1,lon1), (lat2, lon2))

def get_graph_distances(use_graph, ways):
    node_latlon_dict = get_node_latlon_dict(ways)
    edges = use_graph.edges
    out_dist = 0
    for edge in edges: 
        node1 = node_latlon_dict[edge[0]]
        node2 = node_latlon_dict[edge[1]]
        out_dist += get_edge_distance(node1, node2)
    return out_dist

def get_degree_distribution(use_graph): 
    outdict = dict()
    for node in use_graph.nodes: 
        degree = use_graph.degree(node)
        if degree not in outdict.keys(): 
            outdict[degree] = 1
        else: 
            outdict[degree] += 1
    return outdict

def find_intersections(outdict): 
    intersections = 0
    for key in outdict.keys(): 
        if key >= 3: 
            intersections += outdict[key]
    return intersections

# Removes all nodes that have a degree of 2, connects intersections/dead ends directly to each other
def strip_the_paths(use_ingraph):
    ingraph = use_ingraph.copy()
    nodes = list(ingraph.nodes)
    for node in nodes: 
        degree = ingraph.degree(node)
        if degree == 2: 
            neighbors = [i for i in ingraph.neighbors(node)]
            ingraph.remove_node(node)
            ingraph.add_edge(neighbors[0], neighbors[1])
    return ingraph

def get_community_box_area(inshape): 
    box = inshape.bounds
    topleft = (box[0],box[1])
    topright = (box[0],box[3])
    bottomleft = (box[2],box[1])
    bottomright = (box[2],box[3])
    polygon = geometry.box(box[0], box[1], box[2], box[3])
    return area(mapping(polygon))/1000000

def get_community_hull_area(inshape): 
    return area(mapping(inshape.convex_hull))/1000000

def get_edge_distance(node1, node2): 
    lat1 = float(node1['lat'])
    lon1 = float(node1['lon'])
    lat2 = float(node2['lat'])
    lon2 = float(node2['lon'])

    return haversine((lat1,lon1), (lat2, lon2))

In [5]:

#load in data
prefix = '06_26_2026'
#total_result_ways = load_pickle('pickle_folder/'+prefix + '/' + prefix+'_total_result_ways.pkl')
G, total_result_ways = lite_load_pickles(prefix)

# Get lats and lons for all nodes
node_dict = dict() 

for way in total_result_ways: 
    for node in way.nodes: 
        node_dict[node.id] = {'lat': node.lat, 'lon': node.lon}

# Go through all nodes in node_dict, get neighbors in graph, find distance between unique pairs of neighbors, add to list

distance_list = []
distance_pairs = set()

for node in node_dict.keys(): 
    neighbors = [i for i in G.neighbors(node)]
    for neighbor in neighbors: 
        sorted_pair = tuple(sorted([node, neighbor]))
        if sorted_pair not in distance_pairs: 
            edge_distance = get_edge_distance(node_dict[neighbor], node_dict[node])
            distance_list.append(edge_distance)
            distance_pairs.add(sorted_pair)

#allow to compare

median = np.median(np.array(distance_list))
dump_pickle('misc_info','inter_node_median', median)
#input('bring the system monitor over here and compare')


%xdel node_dict
gc.collect()
%xdel distance_pairs
gc.collect()


lcc_ways = make_new_lcc(G, total_result_ways)
#lcc_merged = convert_to_borders(lcc_ways)[0]

NameError: name 'distance_pairs' is not defined


In [6]:
# Go through each way
# check each node: what community does it belong to?
# If 2 or more nodes belong to the same community, add that way to the output community. 

def run_greedy_modularity(G, lcc_ways,resolution = 1,weighted=True,use_n = None):

    lcc_nodes = set()
    for way in lcc_ways:
        for node in way.nodes: 
            lcc_nodes.add(node.id)
    
    lcc_subgraph = G.subgraph(lcc_nodes)
    if weighted: 
        if use_n != None: 
            communities = nx.community.greedy_modularity_communities(lcc_subgraph, resolution=resolution, weight='weight', best_n = use_n, cutoff=use_n)
        else: 
            communities = nx.community.greedy_modularity_communities(lcc_subgraph, resolution=resolution, weight='weight')
    else: 
        if use_n != None: 
            communities = nx.community.greedy_modularity_communities(lcc_subgraph, resolution=resolution, best_n=use_n, cutoff=use_n)
        else: 
            communities = nx.community.greedy_modularity_communities(lcc_subgraph, resolution=resolution)
    communities = sorted(communities, key = lambda x: -len(x))
    
    community_geojson_dict = dict()
    
    for lcc_way_dex in range(len(lcc_ways)):
    #for lcc_way_dex in range(10):
        #print(f"I am on {lcc_way_dex}/{len(lcc_ways)}")
        lcc_way = lcc_ways[lcc_way_dex]
        count_community_dict = dict() 
        for node in lcc_way.nodes: 
            for community_dex in range(len(communities)):
                use_community = communities[community_dex]
                if node.id in use_community: 
                    if community_dex not in count_community_dict.keys(): 
                        count_community_dict[community_dex] = 1
                    else: 
                        count_community_dict[community_dex] += 1
                    break
            # if count_community_dict[community_dex] >= 2: 
            #     print(count_community_dict)
            #     if community_dex not in community_geojson_dict.keys(): 
            #         community_geojson_dict[community_dex] = [lcc_way]
            #     else: 
            #         community_geojson_dict[community_dex].append(lcc_way)
            #     break
        highest_list = sorted([(i, count_community_dict[i]) for i in count_community_dict.keys()], key = lambda x: -x[1])
        #print(highest_list)
        highest = highest_list[0][0]
        if highest not in community_geojson_dict.keys(): 
            community_geojson_dict[highest] = [lcc_way]
        else: 
            community_geojson_dict[highest].append(lcc_way)

    use_map_community_dict = dict()
    for map_community_dex in range(len(list(community_geojson_dict.keys()))):
        use_map_community = community_geojson_dict[map_community_dex] #get the geojson dict from the map_community_dex
        use_map_community_merged = convert_to_borders(use_map_community)[0] #convert that to borders directly
        use_map_community_dict[map_community_dex] = {'shape': use_map_community_merged, 'community': use_map_community}
    return use_map_community_dict, community_geojson_dict

# Create partition files for later loading

## Start with n_list (ie, the community_number)

In [7]:
n_list = list(range(6,22,2))


start_time = time.time()
for use_n in n_list: 
    n_filename = f'{use_n}_communities'
    if n_filename not in os.listdir('partitions/community_number'):
        use_map_community_dict, community_geojson_dict = run_greedy_modularity(G, lcc_ways,resolution=1,weighted=False,use_n=use_n)
        dump_pickle('partitions/community_number', n_filename, [use_map_community_dict, community_geojson_dict])

try:
    print(type(use_map_community_dict))
    %xdel use_map_community_dict
    %xdel community_geojson_dict
    gc.collect()
except: 
    print("There's nothing to worry about here.")
print(f'This took {time.time() - start_time}')

There's nothing to worry about here.
This took 0.0023567676544189453


## Continue with resolution



In [8]:
resolution_list = [i/10000 for i in range(1, 8)]

start_time = time.time()
for use_res in resolution_list: 
    res_filename = f'{use_res}_resolution'
    if res_filename not in os.listdir('partitions/resolution'):
        use_map_community_dict, community_geojson_dict = run_greedy_modularity(G, lcc_ways,resolution=use_res,weighted=False)
        dump_pickle('partitions/resolution', res_filename, [use_map_community_dict, community_geojson_dict])

try:
    print(len(list(use_map_community_dict.keys())))
    %xdel use_map_community_dict
    %xdel community_geojson_dict
    gc.collect()
except: 
    print("There's nothing to worry about here.")
print(f'This took {time.time() - start_time}')

There's nothing to worry about here.
This took 0.002282381057739258


In [9]:
import sys

# Print variables and their sizes in Megabytes (MB)
for var, obj in list(locals().items()):
    if not var.startswith("_"):
        size = sys.getsizeof(obj) / (1024 * 1024)
        if size > 10:  # Only show items larger than 10 MB
            print(f"{var}: {size:.2f} MB")

distance_list: 18.38 MB
lcc_ways: 20.51 MB


In [12]:
try: 
    print(type(G))
    %xdel G
    gc.collect()
except: 
    print("G not present")
try: 
    print(type(lcc_merged))
    %xdel lcc_merged
    gc.collect()
except: 
    print("lcc_merged not present")
try: 
    print(type(lcc_ways))
    %xdel lcc_ways
    gc.collect()
except:
    print("lcc_ways not present")
try:
    print(type(total_result_ways))
    %xdel total_result_ways
    gc.collect()
except: 
    print("total_result_ways not present")
try: 
    print(type(distance_list))
    %xdel distance_list
    gc.collect()
except: 
    print("distance_list not present")
gc.collect()

<class 'networkx.classes.graph.Graph'>
lcc_merged not present
<class 'pandas.Series'>
<class 'list'>
<class 'list'>


0

In [1]:
%reset -f
%config InteractiveShell.cache_size = 0
from convenient_pickle import *
import gc
from haversine import haversine
import numpy as np
import pandas as pd
prefix_bike = '06_26_2026_just_bikes'
total_result_ways_bike = load_pickle('pickle_folder/'+prefix_bike + '/' + prefix_bike+'_total_result_ways.pkl')

median = load_pickle('misc_info/inter_node_median')

In [2]:
def get_av_dist_to_trail(use_map_community_dict, bikeways):
    from shapely.geometry import Point, LineString
    from shapely.strtree import STRtree
    from shapely.ops import nearest_points, linemerge, unary_union
    from shapely.geometry import Point

    community_bike_info_dict = dict()

    typeset = set()


    for community in use_map_community_dict.keys(): 
        #print(total_result_ways_bike)
        bike_way_ids = {i.id for i in bikeways}
        total_result_ways = [i for i in use_map_community_dict[community]['community'] if i.id not in bike_way_ids]
        del bike_way_ids
        gc.collect()

        shapely_bike_ways = []
        for way in bikeways: 
            coord_list = LineString([(i.lon, i.lat) for i in way.nodes])
            shapely_bike_ways.append(coord_list)
        shapely_bike_ways = unary_union(linemerge(shapely_bike_ways))
        #bikeways = shapely_bike_ways
        #del shapely_bike_ways
        gc.collect()
        if shapely_bike_ways.geom_type == "MultiLineString":
            shapely_bike_ways = list(shapely_bike_ways.geoms)
        else:
            shapely_bike_ways = [shapely_bike_ways]
        
        tree = STRtree(shapely_bike_ways)
        dist_to_bike_list = []
        test_normal_dists = []
        
        for way in total_result_ways: 
            nodes = [i for i in way.nodes]
            for node_dex in range(len(nodes)-1): 
                first_node = nodes[node_dex]
                second_node = nodes[node_dex+1] 
                distance = haversine((first_node.lat, first_node.lon), (second_node.lat, second_node.lon))
                midpoint = Point((first_node.lon+second_node.lon)/2, (first_node.lat+second_node.lat)/2)
                dists = []
                n_idx = tree.nearest(midpoint)
                nearest = shapely_bike_ways[n_idx]
                p1, p2 = nearest_points(midpoint, nearest)
                normal_dist = haversine((p1.x, p1.y), (p2.x, p2.y))
                test_normal_dists.append((normal_dist, distance))
                normal_dist = (normal_dist) * distance/median
                dist_to_bike_list.append((normal_dist, distance/median))
        community_bike_info_dict[community] = {'test': test_normal_dists, 'normal_dist': dist_to_bike_list}
    return community_bike_info_dict

In [58]:
resolution_list = [i/10000 for i in range(1, 8)]

for resolution in resolution_list: 
    [use_map_community_dict, community_geojson_dict] = load_pickle(f'partitions/resolution/{resolution}_resolution')

    community_bike_info_dict = get_av_dist_to_trail(use_map_community_dict, total_result_ways_bike)
    save_dict = dict()
    for key in community_bike_info_dict.keys(): 
        save_dict[f'raw_median_{key}'] = np.median([i[0] for i in community_bike_info_dict[key]['test']])
        save_dict[f'raw_average_{key}'] = np.mean([i[0] for i in community_bike_info_dict[key]['test']])
        save_dict[f'representative_average_{key}'] = sum([i[0] for i in community_bike_info_dict[key]['normal_dist']])/sum([i[1] for i in community_bike_info_dict[key]['normal_dist']])
    save_dict = pd.DataFrame.from_dict(save_dict,orient='index').T
    save_dict.to_csv(f'csvs/resolution/{resolution}_resolution/{resolution}_resolution_distance_to_bike_trail.csv')
    del community_bike_info_dict
    del use_map_community_dict
    del community_geojson_dict
    gc.collect()
    print(f'Completed {resolution} resolution.')

Completed 0.0001 resolution.
Completed 0.0002 resolution.
Completed 0.0003 resolution.
Completed 0.0004 resolution.
Completed 0.0005 resolution.
Completed 0.0006 resolution.
Completed 0.0007 resolution.


In [3]:
n_list = list(range(6,22,2))

for use_n in n_list: 
    [use_map_community_dict, community_geojson_dict] = load_pickle(f'partitions/community_number/{use_n}_communities')

    community_bike_info_dict = get_av_dist_to_trail(use_map_community_dict, total_result_ways_bike)
    save_dict = dict()
    for key in community_bike_info_dict.keys(): 
        save_dict[f'raw_median_{key}'] = np.median([i[0] for i in community_bike_info_dict[key]['test']])
        save_dict[f'raw_average_{key}'] = np.mean([i[0] for i in community_bike_info_dict[key]['test']])
        save_dict[f'representative_average_{key}'] = sum([i[0] for i in community_bike_info_dict[key]['normal_dist']])/sum([i[1] for i in community_bike_info_dict[key]['normal_dist']])
    save_dict = pd.DataFrame.from_dict(save_dict,orient='index').T
    save_dict.to_csv(f'csvs/community_number/{use_n}_communities/{use_n}_communities_distance_to_bike_trail.csv')
    del community_bike_info_dict
    del use_map_community_dict
    del community_geojson_dict
    gc.collect()
    print(f'Completed {use_n} communities.')

Completed 6 communities.
Completed 8 communities.
Completed 10 communities.
Completed 12 communities.
Completed 14 communities.
Completed 16 communities.
Completed 18 communities.
Completed 20 communities.
